In [49]:
!pip install matplotlib scipy pandas cvxpy tqdm seaborn openai cvxpy[glpk] polarix -q

zsh:1: no matches found: cvxpy[glpk]


In [50]:
import pickle
import sys
import time                                                                                                                                    
from pathlib import Path
from collections import Counter, defaultdict                                                                                                   
from itertools import combinations

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.optimize import linprog
import pandas as pd

sys.path.insert(0, "/Users/gabesmithline/Desktop/Causal-Game-Analysis")


from src.iterative_game_analysis.metagame import MetaGame

from visuals.visualize_analysis import DISPLAY_NAMES, STRATEGY_ORDER
from evaluation.curb_analysis import *
from evaluation.bootstrap_analysis import load_all_games, compute_payoff_matrix_from_games 
from src.iterative_game_analysis.full_analysis import load_crossplay_to_dataframe                                                              
from src.iterative_game_analysis.bootstrap import Bootstrap 


In [51]:

crossplay_dir = "/Users/gabesmithline/Desktop/Causal-Game-Analysis/data/crossplay"
print(crossplay_dir)
strategy_names = ["walk", "tough", "soft",
                    "openai_5.2_none", "openai_5.2_low", "openai_5.4_low", "ef1_bargainer", "ppo", "psro", "nfsp"] 
df = load_crossplay_to_dataframe(crossplay_dir, strategy_names, raw_utility=True)  
print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")

/Users/gabesmithline/Desktop/Causal-Game-Analysis/data/crossplay
  Loading walk vs walk...
  Loading walk vs tough...
  Loading walk vs soft...
  Loading walk vs openai_5.2_none...
  Loading walk vs openai_5.2_low...
  Loading walk vs openai_5.4_low...
  Loading walk vs ef1_bargainer...
  Loading walk vs ppo...
  Loading walk vs psro...
  Loading walk vs nfsp...
  Loading tough vs walk...
  Loading tough vs tough...
  Loading tough vs soft...
  Loading tough vs openai_5.2_none...
  Loading tough vs openai_5.2_low...
  Loading tough vs openai_5.4_low...
  Loading tough vs ef1_bargainer...
  Loading tough vs ppo...
  Loading tough vs psro...
  Loading tough vs nfsp...
  Loading soft vs walk...
  Loading soft vs tough...
  Loading soft vs soft...
  Loading soft vs openai_5.2_none...
  Loading soft vs openai_5.2_low...
  Loading soft vs openai_5.4_low...
  Loading soft vs ef1_bargainer...
  Loading soft vs ppo...
  Loading soft vs psro...
  Loading soft vs nfsp...
  Loading openai_5.2_none

In [52]:
# Build the average matrices (no bootstrap, just the point estimate)
boot = Bootstrap(df=df, n_samples=1, seed=42, policies=strategy_names)
matrices = boot._build_all_matrices(df, strategy_names)

avg_payoff = matrices["payoff"]       # 10x10, used for equilibrium solving + UW
nw_matrix = matrices["nw"]            # 10x10, Nash welfare per matchup
nw_plus_matrix = matrices["nw_plus"]  # 10x10, Nash welfare on advantages
ef1_matrix = np.nan_to_num(matrices["ef1"])          # 10x10, EF1 frequency per matchup
ef1_plus_matrix = np.nan_to_num(matrices["ef1_plus"])# 10x10, EF1+ frequency per matchup  


print("Empirical Meta-Game (avg payoff):\n")
header = "".join(f"{s[:8]:>10}" for s in strategy_names)
print(f"{'':>10}{header}")
print("-" * (10 + 10 * len(strategy_names)))
for i, name in enumerate(strategy_names):
    row = "".join(f"{avg_payoff[i, j]:>10.2f}" for j in range(len(strategy_names)))
    print(f"{name[:8]}{row}")

Empirical Meta-Game (avg payoff):

                walk     tough      soft  openai_5  openai_5  openai_5  ef1_barg       ppo      psro      nfsp
--------------------------------------------------------------------------------------------------------------
walk    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81
tough    303.81    303.81    579.17    330.69    324.55    326.84    304.62    319.51    317.67    325.16
soft    303.81     50.43    303.59    230.81    229.89    231.06    310.96    197.45    165.56    287.51
openai_5    303.81    298.73    446.43    381.55    382.81    382.91    372.72    358.69    343.24    357.85
openai_5    303.81    302.70    438.95    383.37    382.31    398.28    377.30    364.88    364.36    368.21
openai_5    303.81    299.88    444.60    393.23    378.27    379.37    366.09    358.83    347.96    364.82
ef1_barg    303.81    305.08    394.19    367.14    363.18    367.24    362.15    346.18    345.92  

In [53]:
sigma = MetaGame(payoff_matrix=avg_payoff, policies=strategy_names).solve("lle")
for mass, name in zip(sigma, strategy_names):
    print(f"Strat name: {name}, mass: {mass}")

Strat name: walk, mass: 9.956110439253042e-13
Strat name: tough, mass: 9.956110439253042e-13
Strat name: soft, mass: 9.956110439253042e-13
Strat name: openai_5.2_none, mass: 9.956110439253042e-13
Strat name: openai_5.2_low, mass: 1.0794707895077101e-13
Strat name: openai_5.4_low, mass: 1.0794707895077101e-13
Strat name: ef1_bargainer, mass: 1.2637487162163438e-14
Strat name: ppo, mass: 0.9999999999957636
Strat name: psro, mass: 1.2637487162163438e-14
Strat name: nfsp, mass: 1.2637487162163438e-14


In [54]:
nw_plus_mene = sigma @ nw_plus_matrix @ sigma
print(nw_plus_mene)
nw_mene= sigma @ nw_matrix @ sigma 
print(nw_mene)
ef1__mene = sigma @ ef1_matrix @ sigma
print(ef1__mene)
ef1_plus_mene = sigma @ ef1_plus_matrix @ sigma 
print(ef1_plus_mene)

52.659901863620625
327.861621789006
0.5234899328832839
0.541835552979385


In [57]:
METRIC_NAMES = ["uw", "nw", "nw_plus", "ef1", "ef1_plus"]                                                                                                                 
metric_matrices = {
    "uw": avg_payoff, "nw": nw_matrix, "nw_plus": nw_plus_matrix,                                                                                                         
    "ef1": ef1_matrix, "ef1_plus": ef1_plus_matrix,
}
policy_to_idx = {p: i for i, p in enumerate(strategy_names)}
ablatable = [s for s in strategy_names if s != "walk"]

# Full-game welfare
W_full = {m: float(sigma @ np.nan_to_num(metric_matrices[m], nan=0.0) @ sigma) for m in METRIC_NAMES}
print("Full-game welfare:", {m: f"{v:.4f}" for m, v in W_full.items()})

# Hold-one-out
singleton_effects = {}
for s in ablatable:
    remaining = [q for q in strategy_names if q != s]
    idx = [policy_to_idx[q] for q in remaining]
    sub_mg = MetaGame(remaining, avg_payoff[np.ix_(idx, idx)])
    sigma_sub = sub_mg.solve("mene")
    W_sub = {m: float(sigma_sub @ np.nan_to_num(metric_matrices[m][np.ix_(idx, idx)], nan=0.0) @ sigma_sub) for m in METRIC_NAMES}
    singleton_effects[s] = {m: (W_full[m] - W_sub[m]) / W_full[m] * 100  for m in METRIC_NAMES}   

df_single = pd.DataFrame(singleton_effects).T
df_single = df_single.sort_values("uw", key=abs, ascending=False)
df_single


Full-game welfare: {'uw': '368.9641', 'nw': '327.8616', 'nw_plus': '52.6599', 'ef1': '0.5235', 'ef1_plus': '0.5418'}


/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000016 > 1e-05).
  warnings.warn(


,uw,nw,nw_plus,ef1,ef1_plus
ppo,17.657792,21.008260,100.000000,100.0000,100.000000
psro,17.657792,21.008260,100.000000,100.0000,100.000000
ef1_bargainer,0.441878,0.595415,2.688458,2.1319,2.310506
tough,0.441878,0.595415,2.688458,2.1319,2.310506
soft,0.441878,0.595415,2.688458,2.1319,2.310506
openai_5.2_none,0.441878,0.595415,2.688458,2.1319,2.310506
openai_5.2_low,0.441878,0.595415,2.688458,2.1319,2.310506
openai_5.4_low,0.441878,0.595415,2.688458,2.1319,2.310506
nfsp,0.441878,0.595415,2.688458,2.1319,2.310506


In [56]:
welfare_cache = {}

def get_welfare(exclude_set):
    key = frozenset(exclude_set)
    if key not in welfare_cache:
        remaining = [q for q in strategy_names if q not in exclude_set]
        idx = [policy_to_idx[q] for q in remaining]
        sub_mg = MetaGame(remaining, avg_payoff[np.ix_(idx, idx)])
        sig = sub_mg.solve("mene")
        welfare_cache[key] = {
            m: float(sig @ np.nan_to_num(metric_matrices[m][np.ix_(idx, idx)], nan=0.0) @ sig)
            for m in METRIC_NAMES
        }
    return welfare_cache[key]

welfare_cache[frozenset()] = W_full
for s in ablatable:
    get_welfare({s})

pairs = list(combinations(ablatable, 2))
pair_effects = {}
for a, b in pairs:
    W_no_a = get_welfare({a})
    W_no_b = get_welfare({b})
    W_no_ab = get_welfare({a, b})
    pair_effects[(a, b)] = {m: W_full[m] - W_no_a[m] - W_no_b[m] + W_no_ab[m] for m in METRIC_NAMES}

df_pairs = pd.DataFrame(pair_effects).T
df_pairs.index = [f"{a} x {b}" for a, b in pair_effects.keys()]
df_pairs = df_pairs.sort_values("uw", key=abs, ascending=False)
df_pairs

/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000016 > 1e-05).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.001 (original failure: Solution has regret 0.000766 > 1e-05).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000011 > 1e-05).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000010 > 1e-05).
  warnings.warn(


,uw,nw,nw_plus,ef1,ef1_plus
ppo x psro,143.872657,155.473325,120.696104,1.205919,1.329624
tough x ppo,1.630370,1.952137,1.415740,0.011160,0.012519
tough x psro,1.630370,1.952137,1.415740,0.011160,0.012519
tough x openai_5.2_none,1.630370,1.952137,1.415740,0.011160,0.012519
openai_5.4_low x ef1_bargainer,1.630370,1.952137,1.415740,0.011160,0.012519
ef1_bargainer x psro,1.630370,1.952137,1.415740,0.011160,0.012519
ef1_bargainer x ppo,1.630370,1.952137,1.415740,0.011160,0.012519
ef1_bargainer x nfsp,1.630370,1.952137,1.415740,0.011160,0.012519
openai_5.2_none x ef1_bargainer,1.630370,1.952137,1.415740,0.011160,0.012519
openai_5.4_low x psro,1.630370,1.952137,1.415740,0.011160,0.012519
